## 1. Environment/runtime check

Executer sur le kernel distant PYNQ-Z2 avec le runtime recompile VFPv3. Arreter les anciens notebooks camera/WebRTC et leur serveur 8765 avant de commencer. Un seul client Flutter est accepte pour ce jalon. Executer les cellules dans l'ordre ; ne pas reexecuter la construction pendant une session active.


In [1]:
import sys
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python version:", sys.version)
print("Architecture:", platform.machine())
print("Current working directory:", Path.cwd())
print("Home directory:", Path.home())

for package, label in [("numpy", "NumPy version"),
                       ("tflite-runtime", "tflite-runtime version")]:
    try:
        print(f"{label}: {version(package)}")
    except PackageNotFoundError:
        print(f"{label}: package non trouve dans ce kernel")

import asyncio
import json
import gi
import websockets
from concurrent.futures import ThreadPoolExecutor
from websockets.exceptions import ConnectionClosed

gi.require_version("Gst", "1.0")
gi.require_version("GstWebRTC", "1.0")
gi.require_version("GstSdp", "1.0")
gi.require_version("GstVideo", "1.0")
from gi.repository import Gst, GstWebRTC, GstSdp, GstVideo
Gst.init(None)
print("GStreamer:", Gst.version_string())
print("websockets:", websockets.__version__)


Python version: 3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
Architecture: armv7l
Current working directory: /home/xilinx/jupyter_notebooks
Home directory: /root
NumPy version: 1.21.5
tflite-runtime version: 2.13.0
GStreamer: GStreamer 1.20.1
websockets: 10.3


## 2. TFLite model loading

Chargement et inspection repris de person_detection_camera.ipynb : SSD MobileNet V1, labels COCO, seuil 0.5. Aucun changement du runtime ni des fichiers du modele.


In [2]:
import numpy as np
from tflite_runtime.interpreter import Interpreter

print("NumPy import: OK —", np.__version__)
print("Interpreter import: OK")
print("Runtime module:", Interpreter.__module__)

import time
import cv2

MODEL_DIR = Path("/home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1")
MODEL_PATH = MODEL_DIR / "detect.tflite"
LABEL_PATH = MODEL_DIR / "labelmap.txt"
CONFIDENCE_THRESHOLD = 0.5
for path in (MODEL_PATH, LABEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Fichier introuvable sur le PYNQ : {path}")
print("Model path:", MODEL_PATH)
interpreter = None
try:
    interpreter = Interpreter(model_path=str(MODEL_PATH))
    interpreter.allocate_tensors()
except Exception:
    interpreter = None
    print("Model loading failed. Verifier le fichier TFLite CPU, les operateurs et la memoire.")
    raise

print("Model loading: OK")
print("Tensor allocation: OK")
if interpreter is None:
    raise RuntimeError("Charger et allouer le modele avant d'inspecter ses tenseurs.")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

for group, details in [("Input", input_details), ("Output", output_details)]:
    print(f"\n{group} tensors: {len(details)}")
    for position, tensor in enumerate(details):
        print(f"\n{group} tensor #{position} (index={tensor['index']})")
        print("  name:", tensor["name"])
        print("  shape:", tensor["shape"].tolist())
        print("  shape signature:", tensor.get("shape_signature"))
        print("  dtype:", np.dtype(tensor["dtype"]).name)
        print("  quantization:", tensor["quantization"])
        print("  quantization parameters:", tensor["quantization_parameters"])

expected_shapes = [(1, 10, 4), (1, 10), (1, 10), (1,)]
if len(output_details) != 4:
    raise ValueError("Quatre sorties SSD sont attendues.")
for detail, expected in zip(output_details, expected_shapes):
    if tuple(detail["shape"]) != expected:
        raise ValueError(f"Output shape inattendue pour {detail['name']}: {detail['shape']}")


if not LABEL_PATH.is_file():
    raise FileNotFoundError(f"Label map introuvable sur le PYNQ : {LABEL_PATH}")
labels = [line.strip() for line in LABEL_PATH.read_text(encoding="utf-8-sig").splitlines()]
if labels and labels[0] == "???":
    labels = labels[1:]
if not labels or labels[0] != "person":
    raise ValueError("Label map incompatible : la classe 0 doit etre person.")
if not 0 <= CONFIDENCE_THRESHOLD <= 1:
    raise ValueError("Le seuil doit etre compris entre 0 et 1.")



NumPy import: OK — 1.21.5
Interpreter import: OK
Runtime module: tflite_runtime.interpreter
Model path: /home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1/detect.tflite
Model loading: OK
Tensor allocation: OK

Input tensors: 1

Input tensor #0 (index=175)
  name: normalized_input_image_tensor
  shape: [1, 300, 300, 3]
  shape signature: [  1 300 300   3]
  dtype: uint8
  quantization: (0.0078125, 128)
  quantization parameters: {'scales': array([0.0078125], dtype=float32), 'zero_points': array([128]), 'quantized_dimension': 0}

Output tensors: 4

Output tensor #0 (index=167)
  name: TFLite_Detection_PostProcess
  shape: [1, 10, 4]
  shape signature: [ 1 10  4]
  dtype: float32
  quantization: (0.0, 0)
  quantization parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}

Output tensor #1 (index=168)
  name: TFLite_Detection_PostProcess:1
  shape: [1, 10]
  shape signature: [ 1 10]
  dtype: float32
  quantization: (0.0, 0)
 

## 3. Shared GStreamer pipeline construction

Une seule source v4l2src possede /dev/video0. Le tee separe video et IA avec des queues independantes. Les parametres H.264/RTP et la liaison manuelle a webrtcbin restent ceux du notebook valide. La queue IA et appsink gardent chacun au maximum une frame en attente, en abandonnant les anciennes. Aucun VideoCapture OpenCV. Les autres processus doivent avoir libere la camera : le notebook ne peut pas garantir leur absence.


In [3]:
from media_pipeline import create_media_pipeline

pipeline, webrtc, appsink, ai_queue, camera_source, rtp_src_pad = create_media_pipeline()

def on_connection_state_changed(element, _):
    state = element.get_property("connection-state")
    print("PYNQ WebRTC connection state:", state.value_nick)


def on_ice_connection_state_changed(element, _):
    state = element.get_property("ice-connection-state")
    print("PYNQ ICE connection state:", state.value_nick)


def on_ice_gathering_state_changed(element, _):
    state = element.get_property("ice-gathering-state")
    print("PYNQ ICE gathering state:", state.value_nick)


webrtc.connect(
    "notify::connection-state",
    on_connection_state_changed,
)

webrtc.connect(
    "notify::ice-connection-state",
    on_ice_connection_state_changed,
)

webrtc.connect(
    "notify::ice-gathering-state",
    on_ice_gathering_state_changed,
)

rtp_progress = {"buffers": 0}
def count_rtp(pad, info):
    rtp_progress["buffers"] += 1
    return Gst.PadProbeReturn.OK
rtp_src_pad.add_probe(Gst.PadProbeType.BUFFER, count_rtp)
bus = pipeline.get_bus()

def check_bus():
    message = bus.timed_pop_filtered(0, Gst.MessageType.ERROR | Gst.MessageType.EOS)
    if message is not None:
        if message.type == Gst.MessageType.ERROR:
            error, debug = message.parse_error()
            raise RuntimeError(f"GStreamer: {error}; {debug}")
        raise RuntimeError("GStreamer EOS")

def assert_bounded_buffers():
    assert ai_queue.get_property("max-size-buffers") == 1
    assert int(ai_queue.get_property("leaky")) == 2
    assert appsink.get_property("max-buffers") == 1
    assert appsink.get_property("drop")
    level = ai_queue.get_property("current-level-buffers")
    assert level <= 1
    return level


Camera pipeline created: OK
WebRTC element created: OK
WebRTC sink pad: sink_0
RTP -> WebRTC: ok
Camera sources in this pipeline: 1 (GStreamer only)


## 4. appsink frame validation

Test preliminaire du meme pipeline, sans signaling. Il est remis a NULL apres capture ; la session combinee le redemarrera apres connexion Flutter. Aucune acquisition simultanee. Conversion du buffer mappe en NumPy en respectant stride et offset (padding des lignes), puis copie avant unmap. try-pull-sample attend au maximum 5 secondes. Documentation : [appsink](https://gstreamer.freedesktop.org/documentation/app/appsink.html), [queues](https://gstreamer.freedesktop.org/documentation/coreelements/queue.html).


In [4]:

def pull_frame():
    sample = appsink.emit("try-pull-sample", 5 * Gst.SECOND)
    if sample is None:
        raise RuntimeError("appsink: aucune frame recue sous 5 secondes")
    caps = sample.get_caps()
    structure = caps.get_structure(0)
    if structure.get_string("format") != "BGR":
        raise ValueError(f"Format appsink inattendu : {caps.to_string()}")
    video_info = GstVideo.VideoInfo.new_from_caps(caps)
    if video_info is None:
        raise ValueError("Caps video invalides")
    buffer = sample.get_buffer()
    meta = GstVideo.buffer_get_video_meta(buffer)
    stride = meta.stride[0] if meta is not None else video_info.stride[0]
    offset = meta.offset[0] if meta is not None else video_info.offset[0]
    height, width = video_info.height, video_info.width
    success, mapping = buffer.map(Gst.MapFlags.READ)
    if not success:
        raise RuntimeError("Impossible de mapper le buffer appsink")
    try:
        if stride < width * 3 or len(mapping.data) < offset + (height - 1) * stride + width * 3:
            raise ValueError("Buffer BGR/stride invalide")
        frame = np.ndarray((height, width, 3), dtype=np.uint8,
                           buffer=mapping.data, offset=offset,
                           strides=(stride, 3, 1)).copy()
        pts = int(buffer.pts)
    finally:
        buffer.unmap(mapping)
    return frame, pts

single_frame_bgr = None
try:
    if pipeline.set_state(Gst.State.PLAYING) == Gst.StateChangeReturn.FAILURE:
        raise RuntimeError("Impossible de demarrer la camera GStreamer")
    single_frame_bgr, single_pts = pull_frame()
    check_bus()
    print("appsink frame received: True")
    print("Width/height:", single_frame_bgr.shape[1], single_frame_bgr.shape[0])
    print("dtype:", single_frame_bgr.dtype)
    print("AI queue level:", assert_bounded_buffers())
finally:
    pipeline.set_state(Gst.State.NULL)
    pipeline.get_state(5 * Gst.SECOND)
    print("Preliminary camera test stopped")


appsink frame received: True
Width/height: 640 480
dtype: uint8
AI queue level: 0
Preliminary camera test stopped


## 5. Single-frame inference from appsink

Preprocessing, infer_persons et print_result repris sans changement de la baseline camera. La frame BGR appsink remplace uniquement la frame VideoCapture. Cette inference preliminaire est executee lorsque le pipeline est arrete.


In [5]:
def preprocess_frame(image_bgr):
    if len(input_details) != 1:
        raise ValueError("Un seul tenseur d'entree est attendu.")
    if tuple(input_details[0]["shape"]) != (1, 300, 300, 3):
        raise ValueError(f"Input shape inattendue : {input_details[0]['shape']}")
    if np.dtype(input_details[0]["dtype"]) != np.dtype(np.uint8):
        raise TypeError("Le modele doit attendre des pixels uint8.")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    resized_rgb = cv2.resize(image_rgb, (300, 300), interpolation=cv2.INTER_LINEAR)
    input_tensor = np.expand_dims(resized_rgb, axis=0)
    assert input_tensor.shape == (1, 300, 300, 3)
    assert input_tensor.dtype == np.uint8
    return input_tensor

input_tensor = preprocess_frame(single_frame_bgr)
print("Input shape:", input_tensor.shape)
print("Input dtype:", input_tensor.dtype)

def infer_persons(input_tensor, original_height, original_width):
    interpreter.set_tensor(input_details[0]["index"], input_tensor)
    start_time = time.perf_counter()
    interpreter.invoke()
    inference_time_ms = (time.perf_counter() - start_time) * 1000
    
    boxes = interpreter.get_tensor(output_details[0]["index"])[0]
    classes = interpreter.get_tensor(output_details[1]["index"])[0]
    scores = interpreter.get_tensor(output_details[2]["index"])[0]
    count_value = float(interpreter.get_tensor(output_details[3]["index"])[0])
    if not np.isfinite(count_value) or not count_value.is_integer():
        raise ValueError(f"Nombre de detections invalide : {count_value}")
    num_detections = int(count_value)
    if not 0 <= num_detections <= min(len(boxes), len(classes), len(scores)):
        raise ValueError(f"Nombre de detections hors limites : {num_detections}")
    
    persons = []
    for i in range(num_detections):
        class_value = float(classes[i])
        score = float(scores[i])
        if not np.isfinite(class_value) or not class_value.is_integer():
            continue
        class_id = int(class_value)
        if not 0 <= class_id < len(labels):
            continue
        label = labels[class_id]
        if label != "person" or not np.isfinite(score) or score < CONFIDENCE_THRESHOLD:
            continue
        if not np.all(np.isfinite(boxes[i])):
            continue
        ymin, xmin, ymax, xmax = np.clip(boxes[i], 0.0, 1.0)
        x1 = int(np.floor(xmin * original_width))
        y1 = int(np.floor(ymin * original_height))
        x2 = int(np.ceil(xmax * original_width))
        y2 = int(np.ceil(ymax * original_height))
        if x2 <= x1 or y2 <= y1:
            continue
        persons.append({
            "label": label, "confidence": score,
            "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
        })
    return inference_time_ms, num_detections, persons


def print_result(number, elapsed_ms, count, persons):
    print(f"\nInference number: {number}")
    print(f"Inference time: {elapsed_ms:.2f} ms")
    print(f"Total detections returned: {count}")
    print(f"Persons detected: {len(persons)}")
    for person_number, person in enumerate(persons, start=1):
        box = person["bbox"]
        print(f"Person {person_number}: {person['label']}, confidence={person['confidence']:.1%}")
        print(f"bbox: x1={box['x1']}, y1={box['y1']}, x2={box['x2']}, y2={box['y2']}")

single_frame_validated = False
single_time_ms, single_count, single_persons = infer_persons(
    input_tensor, *single_frame_bgr.shape[:2]
)
print_result("single-frame test", single_time_ms, single_count, single_persons)
single_frame_validated = True


Input shape: (1, 300, 300, 3)
Input dtype: uint8

Inference number: single-frame test
Inference time: 1000.63 ms
Total detections returned: 10
Persons detected: 1
Person 1: person, confidence=68.8%
bbox: x1=5, y1=126, x2=430, y2=477


## 6. WebRTC signaling startup

Memes messages offer/answer/ice, serveur 0.0.0.0:8765 et callbacks SDP/ICE que la baseline. Le serveur est ouvert ici et handle_client demarre immediatement le pipeline a la connexion Flutter. La section 7 attend seulement la connexion WebRTC avant de demarrer les inferences. Ouvrir Flutter puis executer la cellule suivante. Un seul client par session ; pour refaire un essai, reexecuter les sections 3 a 7 apres l'arret complet. Les resultats IA sont publies uniquement sur le serveur metadata independant 0.0.0.0:8766. Chaque client dispose d'une file bornee a un resultat ; un client lent perd les anciens resultats sans bloquer l'inference ou WebRTC.


In [ ]:
import asyncio
import json
import websockets

clients = set()
pending_sends = set()
client_gone = asyncio.Event()
event_loop = asyncio.get_running_loop()


async def broadcast(message):
    if not clients:
        return

    dead = []

    for websocket in clients:
        try:
            await websocket.send(message)
        except Exception:
            dead.append(websocket)

    for websocket in dead:
        clients.discard(websocket)


def send_json_from_gstreamer(payload):
    message = json.dumps(payload)

    future = asyncio.run_coroutine_threadsafe(
        broadcast(message),
        event_loop,
    )
    pending_sends.add(future)
    future.add_done_callback(pending_sends.discard)


def on_ice_candidate(element, mline_index, candidate):
    print(
        f"PYNQ local ICE: "
        f"mline={mline_index} candidate={candidate}"
    )

    send_json_from_gstreamer({
        "type": "ice",
        "candidate": candidate,
        "sdpMLineIndex": mline_index,
    })


def on_offer_created(promise, _, __):
    promise.wait()

    reply = promise.get_reply()
    offer = reply.get_value("offer")

    webrtc.emit("set-local-description", offer, None)

    sdp_text = offer.sdp.as_text()

    print("SDP offer created")

    send_json_from_gstreamer({
        "type": "offer",
        "sdp": sdp_text,
    })


def on_negotiation_needed(element):
    print("Negotiation needed")

    promise = Gst.Promise.new_with_change_func(
        on_offer_created,
        None,
        None,
    )

    element.emit("create-offer", None, promise)


webrtc.connect(
    "on-negotiation-needed",
    on_negotiation_needed,
)

webrtc.connect(
    "on-ice-candidate",
    on_ice_candidate,
)

print("WebRTC callbacks configured")
pipeline_started = False
stopping = False

from websockets.exceptions import ConnectionClosed

async def handle_client(websocket, path):
    global pipeline_started

    if clients:
        await websocket.close(code=1013, reason="One Flutter client at a time")
        return
    print("Flutter client connected")
    clients.add(websocket)

    try:
        if stopping:
            return
        if not pipeline_started:
            state = pipeline.set_state(Gst.State.PLAYING)
            print("Pipeline state:", state.value_nick)
            if state == Gst.StateChangeReturn.FAILURE:
                raise RuntimeError("GStreamer startup failed")
            pipeline_started = True

        async for raw_message in websocket:
            message = json.loads(raw_message)

            print("Received:", message.get("type"))

            if message["type"] == "answer":
                sdp_text = message["sdp"]

                _, sdp = GstSdp.SDPMessage.new()

                result = GstSdp.sdp_message_parse_buffer(
                    sdp_text.encode(),
                    sdp,
                )

                if result != GstSdp.SDPResult.OK:
                    raise RuntimeError("Unable to parse SDP answer")

                answer = GstWebRTC.WebRTCSessionDescription.new(
                    GstWebRTC.WebRTCSDPType.ANSWER,
                    sdp,
                )

                promise = Gst.Promise.new()

                webrtc.emit(
                    "set-remote-description",
                    answer,
                    promise,
                )

                promise.interrupt()

                print("Remote SDP answer applied")

            elif message["type"] == "ice":
                print(
                    "Flutter ICE received:",
                    message.get("candidate"),
                )

                webrtc.emit(
                    "add-ice-candidate",
                    message["sdpMLineIndex"],
                    message["candidate"],
                )

                print("Remote ICE candidate added")

    except ConnectionClosed:
        print("Flutter WebSocket closed")

    finally:
        clients.discard(websocket)
        client_gone.set()
        print("Flutter client disconnected")
signaling_server = await websockets.serve(
    handle_client,
    "0.0.0.0",
    8765,
)

print("Signaling server running on ws://0.0.0.0:8765")

# Metadata only: never send detections through the signaling clients on 8765.
metadata_clients = {}
metadata_server = None


async def send_metadata(websocket, queue):
    try:
        while True:
            payload = await queue.get()
            try:
                await asyncio.wait_for(websocket.send(payload), timeout=2)
            finally:
                queue.task_done()
    except (ConnectionClosed, asyncio.TimeoutError):
        pass
    except Exception as error:
        print("Metadata send error (pipeline continues):", error)
    finally:
        await websocket.close()


async def handle_metadata_client(websocket, path=None):
    queue = asyncio.Queue(maxsize=1)
    metadata_clients[websocket] = queue
    sender = asyncio.create_task(send_metadata(websocket, queue))
    print("Metadata client connected")
    try:
        await websocket.wait_closed()
    finally:
        metadata_clients.pop(websocket, None)
        sender.cancel()
        await asyncio.gather(sender, return_exceptions=True)
        print("Metadata client disconnected; pipeline continues")


def publish_detection(width, height, inference_ms, persons):
    # Called on the asyncio loop after the worker completes; no network wait.
    payload = json.dumps({
        "type": "detection",
        "frame": {"width": int(width), "height": int(height)},
        "inference_ms": float(inference_ms),
        "persons": persons,
    }, allow_nan=False)
    for queue in list(metadata_clients.values()):
        if queue.full():
            queue.get_nowait()
            queue.task_done()
        queue.put_nowait(payload)


try:
    metadata_server = await websockets.serve(
        handle_metadata_client, "0.0.0.0", 8766, close_timeout=1,
    )
    print("Metadata server running on ws://0.0.0.0:8766")
except OSError as error:
    print("Metadata server unavailable (video/inference can continue):", error)


WebRTC callbacks configured
Signaling server running on ws://0.0.0.0:8765
Metadata server running on ws://0.0.0.0:8766


Metadata client connected
Flutter client connected
Pipeline state: success
Negotiation needed
SDP offer created
PYNQ ICE gathering state: gathering
PYNQ local ICE: mline=0 candidate=candidate:1 1 UDP 2015363327 192.168.2.99 53562 typ host
PYNQ local ICE: mline=0 candidate=candidate:2 1 TCP 1015021823 192.168.2.99 9 typ host tcptype active
PYNQ local ICE: mline=0 candidate=candidate:3 1 TCP 1010827519 192.168.2.99 37521 typ host tcptype passive
PYNQ local ICE: mline=0 candidate=candidate:4 1 UDP 2015363583 192.168.200.111 56125 typ host
PYNQ local ICE: mline=0 candidate=candidate:5 1 TCP 1015022079 192.168.200.111 9 typ host tcptype active
PYNQ local ICE: mline=0 candidate=candidate:6 1 TCP 1010827775 192.168.200.111 37059 typ host tcptype passive
PYNQ local ICE: mline=0 candidate=candidate:7 1 UDP 2015363839 fe80::3440:20ff:fe6c:9b78 52199 typ host
PYNQ local ICE: mline=0 candidate=candidate:8 1 TCP 1015022335 fe80::3440:20ff:fe6c:9b78 9 typ host tcptype active
PYNQ local ICE: mline=0 

## 7. Combined WebRTC + detection run

Dix inferences par defaut. Un unique worker execute capture appsink + inference sequentiellement : aucun travail IA dans les callbacks GStreamer, aucun empilement de taches. La boucle asyncio reste disponible pour le signaling. Cela evite le blocage par les files, sans garantir l'absence de contention CPU sur le Cortex-A9. Apres connexion, les etats WebRTC/ICE, le progres RTP, les timestamps appsink et la queue sont controles a chaque inference. Le progres RTP ne prouve pas a lui seul l'affichage dans Chrome : verifier aussi Flutter visuellement. Arret complet en finally, y compris sur erreur ou interruption. Si vous abandonnez apres la section 6, fermer les serveurs signaling et metadata avant un nouvel essai.


In [7]:

MAX_INFERENCES = 50
CONNECT_TIMEOUT_SECONDS = 60
inference_times_ms = []
status_records = []
worker = None
pending_job = None
loop_start = time.perf_counter()

def process_latest_frame():
    frame, pts = pull_frame()
    tensor = preprocess_frame(frame)
    elapsed, count, persons = infer_persons(tensor, *frame.shape[:2])
    return pts, elapsed, count, persons, frame.shape[1], frame.shape[0]

try:
    if not single_frame_validated:
        raise RuntimeError("Valider la section 5 avant le fonctionnement combine")
    if not isinstance(MAX_INFERENCES, int) or MAX_INFERENCES <= 0:
        raise ValueError("MAX_INFERENCES doit etre positif")
    deadline = time.monotonic() + CONNECT_TIMEOUT_SECONDS
    while webrtc.get_property("connection-state").value_nick != "connected":
        check_bus()
        if client_gone.is_set():
            raise RuntimeError("Flutter s'est deconnecte")
        if time.monotonic() > deadline:
            raise TimeoutError("Connexion WebRTC non etablie sous 60 secondes")
        await asyncio.sleep(0.1)
    print("WebRTC connected; combined run starting")
    worker = ThreadPoolExecutor(max_workers=1)
    previous_pts = None
    for number in range(1, MAX_INFERENCES + 1):
        rtp_before = rtp_progress["buffers"]
        pending_job = asyncio.get_running_loop().run_in_executor(worker, process_latest_frame)
        pts, elapsed, count, persons, width, height = await asyncio.shield(pending_job)
        pending_job = None
        publish_detection(width, height, elapsed, persons)
        check_bus()
        connection = webrtc.get_property("connection-state").value_nick
        ice = webrtc.get_property("ice-connection-state").value_nick
        level = assert_bounded_buffers()
        progressed = rtp_progress["buffers"] > rtp_before
        fresh = pts != previous_pts if pts != Gst.CLOCK_TIME_NONE else None
        previous_pts = pts
        inference_times_ms.append(elapsed)
        status_records.append((connection, ice, progressed, fresh, level))
        print_result(number, elapsed, count, persons)
        print(f"WebRTC={connection}; ICE={ice}; RTP progressed={progressed}")
        print(f"appsink frame received; PTS={pts}; new PTS={fresh}; AI queue={level}/1; appsink max=1 drop=True")
        print("Camera source count: 1; OpenCV camera opens: 0")
        if connection != "connected" or client_gone.is_set():
            raise RuntimeError("WebRTC n'est plus actif pendant l'inference")
        if not progressed:
            print("ATTENTION: aucun progres RTP observe pendant cette inference")
except (KeyboardInterrupt, asyncio.CancelledError):
    print("Combined run interrupted")
finally:
    stopping = True
    # Stop acquisition first, so a waiting appsink pull is released.
    pipeline.set_state(Gst.State.NULL)
    if worker is not None:
        worker.shutdown(wait=True)
    if pending_job is not None:
        try:
            await asyncio.shield(pending_job)
        except Exception as error:
            print("Worker stopped:", error)
    for future in list(pending_sends):
        future.cancel()
    for client in list(clients):
        await client.close()
    # Allow the last result to flush, bounded even for slow metadata clients.
    if metadata_server is not None:
        try:
            await asyncio.wait_for(
                asyncio.gather(*(queue.join() for queue in list(metadata_clients.values()))),
                timeout=2,
            )
        except asyncio.TimeoutError:
            print("Metadata flush timeout; continuing shutdown")
        metadata_server.close()
        await metadata_server.wait_closed()
    signaling_server.close()
    await signaling_server.wait_closed()
    pipeline.set_state(Gst.State.NULL)
    pipeline.get_state(5 * Gst.SECOND)
    loop_elapsed_seconds = time.perf_counter() - loop_start
    print("Camera, worker and signaling stopped")


WebRTC connected; combined run starting

Inference number: 1
Inference time: 2285.13 ms
Total detections returned: 10
Persons detected: 1
Person 1: person, confidence=71.1%
bbox: x1=52, y1=139, x2=621, y2=480
WebRTC=connected; ICE=completed; RTP progressed=True
appsink frame received; PTS=81492193510; new PTS=True; AI queue=0/1; appsink max=1 drop=True
Camera source count: 1; OpenCV camera opens: 0

Inference number: 2
Inference time: 1912.00 ms
Total detections returned: 10
Persons detected: 1
Person 1: person, confidence=68.0%
bbox: x1=66, y1=177, x2=592, y2=474
WebRTC=connected; ICE=completed; RTP progressed=True
appsink frame received; PTS=84025476079; new PTS=True; AI queue=0/1; appsink max=1 drop=True
Camera source count: 1; OpenCV camera opens: 0

Inference number: 3
Inference time: 2017.37 ms
Total detections returned: 10
Persons detected: 1
Person 1: person, confidence=68.8%
bbox: x1=33, y1=147, x2=603, y2=474
WebRTC=connected; ICE=completed; RTP progressed=True
appsink frame 

## 8. Performance/status summary

Bilan des inferences terminees durant la session combinee. Les taux calcules sur invoke() excluent capture, conversion, signaling et affichage. La queue/appsink sont bornes par configuration ; les frames abandonnees ne sont pas comptees sur GStreamer 1.20. La validation reelle exige de voir la video continuer dans Flutter pendant ces mesures.


In [ ]:

print("Total inference count:", len(inference_times_ms))
if inference_times_ms:
    mean_ms = sum(inference_times_ms) / len(inference_times_ms)
    print(f"Average inference time: {mean_ms:.2f} ms")
    print(f"Min/max inference time: {min(inference_times_ms):.2f} / {max(inference_times_ms):.2f} ms")
    print(f"Approximate inference FPS: {1000 / mean_ms:.3f}" if mean_ms > 0 else "FPS unavailable")
else:
    print("Aucune inference combinee terminee")
print("WebRTC connected at all recorded checks:", bool(status_records) and all(r[0] == "connected" for r in status_records))
print("RTP progressed at all recorded checks:", bool(status_records) and all(r[2] for r in status_records))
print("appsink frames processed:", len(status_records))
print("Max observed AI queue level:", max((r[4] for r in status_records), default=0))
print("Configured buffering: AI queue <= 1, appsink <= 1, dropping old frames")
print("One GStreamer camera source, no OpenCV VideoCapture")
print("Final pipeline state:", pipeline.get_state(0)[1].value_nick)
